## Outlier analysis

This section looks at the traits of just the outliers of the different hierarchies' error distributions

In [ ]:
import os, pathlib
while pathlib.Path.cwd().name != 'forecast_research':
    os.chdir('..') # Run from repo root

import pandas as pd
import numpy as np
import plotly.express as px
import python.utils as utils
import python.plots as plots
from python.time_series import BASE_FORECASTS, RECONCILED_FORECASTS, TimeSeriesType
from python.aggregation import MONTHS, FORTNIGHTS, WEEKS, WEEKDAYS, WEEKENDS, DAYS
from python.temporal_hierarchy import TemporalHierarchy, MinimalTemporalHierarchy, SemanticTemporalHierarchy, BinaryTemporalHierarchy, FullTemporalHierarchy
from python.error_metrics import ErrorMetrics, MergedErrorMetrics, MetricsIndex, LoadedErrorMetrics, ErrorDifference, FilteredErrorMetrics, ClonedErrorMetrics, CustomMetrics
from python.error_metrics import MetricsIndex as MI # Alias
from python.forecast_reconciliation import ForecastReconciliation

In [ ]:
em = LoadedErrorMetrics('metrics/binary_hier_errors.csv')
inc = pd.read_csv('metrics/base_hier_inconsistency.csv', index_col=0)

sth = SemanticTemporalHierarchy()
mth = MinimalTemporalHierarchy()
bth = BinaryTemporalHierarchy()
fth = FullTemporalHierarchy()

error_metric = 'nrmse'
em = FilteredErrorMetrics(em, MI(error_metric='nrmse'))

diffs = MergedErrorMetrics([
    ErrorDifference(em, MI(forecast_type=BASE_FORECASTS, hierarchy=mth), MI(forecast_type=RECONCILED_FORECASTS, hierarchy=fth)),
    ErrorDifference(em, MI(forecast_type=BASE_FORECASTS, hierarchy=mth), MI(forecast_type=RECONCILED_FORECASTS, hierarchy=bth)),
    ErrorDifference(em, MI(forecast_type=BASE_FORECASTS, hierarchy=mth), MI(forecast_type=RECONCILED_FORECASTS, hierarchy=sth)),
    ErrorDifference(em, MI(forecast_type=BASE_FORECASTS, hierarchy=mth), MI(forecast_type=RECONCILED_FORECASTS, hierarchy=mth)),
])

nrmse_em, nrmse_diffs = em, diffs

In [ ]:
nrmse_outliers_em = CustomMetrics.keep_outliers(nrmse_em, 0.95, log=True)

### Statistics

I look at the statistics of the top 5% outlier error values per hierarchy and granularity, on log scale.

In [ ]:
nrmse_outliers_em.get_error_stats(MI(granularity=DAYS))

When looking at the daily granularity log error outliers, `Full` has the smallest median and mean. All other medians under the 90th favor `Full`. The maximum error of `Full` is still extreme, but one order of magnitude smaller than that of `Binary`, and several orders of magnitude smaller than `Semantic`'s

In [ ]:
nrmse_outliers_em.get_error_stats(MI(granularity=MONTHS))

When looking at the monthly granularity log error outliers, `Full` has the lowest percentiles out of all the reconciled error outliers, and also the smallest mean out of these, however the base forecast errors still perform the best in this regard.

### Conclusions

**Full**
- Pros
    - When looking at daily granularity log error outliers, this hierarchy had the smallest median, mean, and lowest percentiles, across all reconciled hierarchies.
    - When looking at monthly granularity log error outliers, this hierarchy had the smallest percentiles and mean, across all reconciled hierarchies.
- Cons
    - When looking at monthly granularity log error outliers, the base forecast outliers still performed better than any reconciled hierarchy, including `Full`.